# Load Packages and Data

In [ ]:
from gp import GaussianProcess
import pandas as pd
import jax
import jax.numpy as jnp
import jax.random as jr
import jax.scipy as jsp
import numpy as np

from functools import partial

import time
import matplotlib.pyplot as plt
import dill

%load_ext autoreload

In [ ]:
%autoreload 2

jax.config.update('jax_enable_x64', False)

y_train = jnp.load('data/y_train.npy')
n = y_train.shape[0]

X_train = jnp.load('data/X_train.npy')
y_train = y_train

obs_count = jnp.load('data/obs_count.npy')

# y_mean = y_train.mean()
# y_std = y_train.std()
# y_train = (y_train - y_mean) / y_std

# Configure and Run GP

In [ ]:
indoor_mask = X_train[:,2].astype(bool)
outdoor_mask = ~X_train[:,2].astype(bool)

indoor_mean = y_train[indoor_mask].mean()
outdoor_mean = y_train[outdoor_mask].mean()

def m(obs):
    return obs[2] * indoor_mean + (1 - obs[2]) * outdoor_mean


ls_xy, ls_z, os_xyz, ls_t, os_t = [0.025, 0.05, 60.0, 0.05, 30.0]
ls_xyz = jnp.array([ls_xy, ls_xy, ls_z])

def K(obs1, obs2):
    d_xyz = (obs1[:3] - obs2[:3])**2
    d_t = (obs1[3] - obs2[3])**2
    return os_xyz * jnp.exp(- (d_xyz / (2*ls_xyz**2)).sum()) + \
        os_t * jnp.exp(- d_t / (2*ls_t**2))

In [ ]:
GP = GaussianProcess(m, K)
GP.fit(X_train, y_train, obs_count)

In [ ]:
K_new = Kn_new = GP.K(X_train, X_train)
m, c = GP.posterior_mean_cov(X_train, jnp.array([28, 48]), K_new, Kn_new)

In [ ]:
@jax.jit
def cholesky_sample(m, c):
    L = jnp.linalg.cholesky(c + jnp.eye(c.shape[0]) * 1e-3)
    return m + L @ jr.normal(jr.PRNGKey(305), shape=m.shape)

@jax.jit
def eigh_sample(m, c):
    eigvals, eigvecs = jnp.linalg.eigh(c)
    eigvals_safe = jnp.maximum(eigvals, 0.0)   # clip negative eigenvalues
    return eigvecs @ (jnp.sqrt(eigvals_safe) * jr.normal(jr.PRNGKey(305), shape=eigvals.shape)) + m
    
jr_sample = jax.jit(lambda m, c: jr.multivariate_normal(jr.PRNGKey(305), m, c+jnp.eye(m.shape[0])*1e-3, method='cholesky'))

In [ ]:
evals, evecs = jnp.linalg.eigh(c)

In [ ]:
(evals < 0).sum()

In [ ]:
evals[evals < 0]

In [ ]:
jnp.linalg.cholesky(c + jnp.eye(c.shape[0]) * 1e-3)

In [ ]:
A = cholesky_sample(m, c).block_until_ready()
B = eigh_sample(m, c).block_until_ready()
C = jr_sample(m, c).block_until_ready()

In [ ]:
%timeit cholesky_sample(m, c).block_until_ready()

In [ ]:
%timeit eigh_sample(m, c).block_until_ready()

In [ ]:
%timeit jr_sample(m, c).block_until_ready()

In [ ]:
c

In [ ]:
def predict(self, X_new, cov_chains, method='parallel'):
    # note: set t_new to be infinity to kill temporal covariance
    K_new = self.K(X_new, X_new.at[:,3].set(jnp.inf))
    Kn_new = self.K(self.X_train, X_new.at[:,3].set(jnp.inf))

    @jax.jit
    def single_posterior(variances):
        m, cov = self.posterior_mean_cov(X_new, variances, K_new, Kn_new)
        return m, jnp.diag(cov)  # (n,) instead of (n, n)

    if method == 'parallel':
        flat_chains = jnp.concatenate(cov_chains, axis=0)
        means, vars = jax.vmap(single_posterior)(flat_chains)
    elif method == 'sequential':
        batch_means, batch_vars = jax.vmap(lambda chain: jax.lax.map(single_posterior, chain))(cov_chains)
        means = jnp.concatenate(batch_means, axis=0)
        vars = jnp.concatenate(batch_vars, axis=0)
    return means, vars

In [ ]:
short_chains = cov_chains[:,:10,]
start = time.time()
result1 = predict(GP, X_train, short_chains, method='parallel')
mid = time.time()
result2 = predict(GP, X_train, short_chains, method='sequential')
end = time.time()
print(f"Par: {mid - start:.4f}")
print(f"Seq: {end - mid:.4f}")

In [ ]:
jnp.allclose(result1[0], result2[0])

In [ ]:
jnp.allclose(result1[1], result2[1])

In [ ]:
result3 = predict(GP, X_train, cov_chains, method='parallel')

In [ ]:
start = time.time()
chain = GP.gibbs(chains=2, samples=500)
end = time.time()
print(f"Elapsed: {end - start}")

In [ ]:
chain

In [ ]:
indoor_vars = chain[1][:,:,1]
outdoor_vars = chain[1][:,:,0]

In [ ]:
plt.plot(range(indoor_vars.shape[1]), indoor_vars[0], label='indoor')
plt.plot(range(outdoor_vars.shape[1]), outdoor_vars[0], label='outdoor')
plt.legend()
plt.show()

In [ ]:
cov_chains = chain[1][:,10::,:]
cov_chains.shape

# Predict on New Data

In [ ]:
X_new = jnp.load('../data/X_test.npy')
X_new = X_new.at[:,3].set(100)
with open('../data/osm_context.pkl', 'rb') as f:
    osm_context = dill.load(f)
wlon_train, wlat_train = np.load('../data/world_train.npy').T
wlon_test, wlat_test = np.load('../data/world_test.npy').T

In [ ]:
new_means, new_vars = GP.predict(X_new, cov_chains, method='sequential')

In [ ]:
vmax = jnp.quantile(y_train, 0.975)
vmin = jnp.quantile(y_train, 0.025)

base_fig, base_ax, osm_metadata = osm_context.generate_base_axis(draw_buildings=False)

sc = base_ax.scatter(wlon_test, wlat_test, c=new_means.mean(axis=0), cmap="RdYlGn", vmin=vmin, vmax=vmax)

tr = base_ax.scatter(wlon_train, wlat_train, c=y_train, cmap="RdYlGn",
                vmin=vmin, vmax=vmax, alpha=1)
plt.ticklabel_format(style='plain', axis='both', useOffset=False)

plt.colorbar(tr)
plt.tight_layout()

# plt.savefig('gridsearch1_best_plot.png', dpi=400)